In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("wikipedia_cities_full_clean.csv")

In [4]:
df

,City,Population,Coordinates,Province/State,Area (km²),Latitude,Longitude,Page URL
0,Karachi,55396.0,"24.86000, 67.01000",Sindh,2160.0,24.860000,67.010000,https://en.wikipedia.org/wiki/Karachi
1,Sindh,NaN,"26.350, 68.850",NaN,NaN,26.350000,68.850000,https://en.wikipedia.org/wiki/Sindh
2,Lahore,NaN,"31.54972, 74.34361",Punjab,NaN,31.549720,74.343610,https://en.wikipedia.org/wiki/Lahore
3,Faisalabad,NaN,"31.41667, 73.09111",Punjab,NaN,31.416670,73.091110,https://en.wikipedia.org/wiki/Faisalabad
4,Rawalpindi,NaN,"33.600, 73.033",Babar Sarfraz Alpa (BPS-20 PSP),NaN,33.600000,73.033000,https://en.wikipedia.org/wiki/Rawalpindi
...,...,...,...,...,...,...,...,...
137,Kotli,NaN,NaN,Azad Kashmir,NaN,NaN,NaN,https://en.wikipedia.org/wiki/Kotli
138,Rawalakot,NaN,"33.8534056, 73.7514750",Azad Kashmir,NaN,33.853406,73.751475,https://en.wikipedia.org/wiki/Rawalakot
139,Gilgit-Baltistan,NaN,"35.35, 75.9",NaN,NaN,35.350000,75.900000,https://en.wikipedia.org/wiki/Gilgit-Baltistan
140,Gilgit,NaN,"35.92083, 74.30833",NaN,NaN,35.920830,74.308330,https://en.wikipedia.org/wiki/Gilgit


In [5]:
df.isnull().sum()

City                0
Population        138
Coordinates         4
Province/State      7
Area (km²)        141
Latitude            4
Longitude           4
Page URL            0
dtype: int64

In [7]:
df = df[df['Latitude'].notna()]

In [8]:
df.isnull().sum()

City                0
Population        135
Coordinates         0
Province/State      7
Area (km²)        137
Latitude            0
Longitude           0
Page URL            0
dtype: int64

In [10]:
import folium
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import numpy as np

In [11]:
mapa_original = folium.Map(location = [df['Latitude'].mean(), df['Longitude'].mean()], zoom_start = 5)

In [14]:
for _, row in df.iterrows():
    folium.CircleMarker(
        location = [row['Latitude'], row['Longitude']],
        radius = 4,
        color = 'blue',
        fill = True,
        fill_opacity = 0.6,
        tooltip = row['City']
    ).add_to(mapa_original)
    
mapa_original

In [16]:
df_cluster = df.copy()

coords = df_cluster[['Latitude','Longitude']].values
coords_scaled = StandardScaler().fit_transform(coords)

In [18]:
db = DBSCAN(eps = 0.3, min_samples = 5).fit(coords_scaled)

In [19]:
df_cluster['Cluster'] = db.labels_

In [21]:
mapa_cluster = folium.Map(location = [df['Latitude'].mean(), df['Longitude'].mean()], zoom_start = 5)

In [22]:
colors = np.array([
    "#%06x" % np.random.randint(0, 0xFFFFFF) for _ in range(len(set(df_cluster['Cluster'])))
])

In [23]:
colors

array(['#5adff7', '#244629', '#6211fd', '#845d85'], dtype='<U7')

In [24]:
for _, row in df_cluster.iterrows():
    color = 'black' if row['Cluster'] == -1 else colors[int(row['Cluster'])]
    folium.CircleMarker(
        location = [row['Latitude'], row['Longitude']],
        radius = 5,
        color = color,
        fill = True,
        fill_opacity = 0.8,
        tooltip = f"{row['City']} (Cluster : {row['Cluster']})"
    ).add_to(mapa_cluster)
    
mapa_cluster